In [1]:
import pandas as pd
df = pd.read_csv('../data/train.csv')

In [2]:
df.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [3]:
df = df.iloc[:, 1:] # dropping the id column

In [4]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [5]:
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
# encoding
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# plotting
import matplotlib.pyplot as plt
import seaborn as sns


In [7]:
# 1. Handle the missing values:- 
# - Drop, Impute (Mean, median, mode, Custering fill ...)
# - Our data doesn't have any missing values.

# 1.1 Handle the duplicates:- 
# - We don't have any duplicates, otherwise we needed to drop.

In [8]:
# 2. Outliers (for numerical columns)
def remove_outlier(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5* iqr
    return df[(df[col] >= lower) & (df[col] <= upper)]

numerical_cols = ['age', 'balance', 'duration', 'campaign', 'previous']
for col in numerical_cols:
    df = remove_outlier(df, col)


In [9]:
# Feature engineering:- 
df['balance_per_age'] = df['balance']/df['age']
df['campaign_per_previous'] = df['campaign'] / (df['previous'] + 1)
df['has_previous_contact'] = (df['pdays'] != -1).astype(int)
df['duration_minutes'] = df['duration'] / 60

In [10]:
# Encoding categorical variables.
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 
                   'contact', 'month', 'poutcome']
# 1-hot encoding for multi-class categories:- 
df_encoded = pd.get_dummies(df, columns = categorical_cols, drop_first=True)


In [11]:
# label encoding for binary categories:
le = LabelEncoder()
df['y'] = le.fit_transform(df['y'])

In [12]:
# Feature selection:-
# correlation analysis
correlation = df_encoded.corr()['y'].abs().sort_values(ascending=False)
print(correlation.head(10))

y                   1.000000
duration_minutes    0.316973
duration            0.316973
month_mar           0.182757
balance_per_age     0.150519
balance             0.139626
housing_yes         0.138760
month_oct           0.134987
contact_unknown     0.128674
month_sep           0.107816
Name: y, dtype: float64


In [13]:
# Selecting features based on the threshold of correlation.
selected_features = correlation[correlation > 0.1].index.to_list()
selected_features.remove('y')

In [14]:
selected_features

['duration_minutes',
 'duration',
 'month_mar',
 'balance_per_age',
 'balance',
 'housing_yes',
 'month_oct',
 'contact_unknown',
 'month_sep',
 'job_student']

In [15]:
# Data splitting:- ## 30% data dropped due to outlier removal.
X = df_encoded[selected_features]
y = df_encoded['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42, stratify=y)

In [16]:
# Feature Scaling:- 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
#.6 Model Building & Training:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42, probability=True),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

In [ ]:
# Train and evaluate models:
results = {}
for name, model in models.items():
    if name in ['Logistic Regression', 'SVM']: # distance prone models
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:,1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    accuracy = model.score(X_test if name not in ['Logistic Regression', 'SVM'] else X_test_scaled, y_test)
    
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {'accuracy': accuracy, 'auc': auc}
    print(f"{name}: Accuracy={accuracy:.3f}, AUC={auc:.3f}")
 

Logistic Regression: Accuracy=0.958, AUC=0.930
Random Forest: Accuracy=0.962, AUC=0.937


In [ ]:
# Hyperparameter tuning:
## Random Forest
param_grid = {
    'n_estimator': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), 
                       param_grid, cv = 5, scoring = 'roc_auc', n_jobs = -1)

rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
print(f"Best RF parameter: {rf_grid.best_params_}")


In [ ]:
# 8. Model evaluation: best modle prediction
y_pred_best = best_rf.predict(X_test)
y_prob_best = best_rf.predict_proba(X_test)[:, 1]

# classification report

print(classification_report(y_test, y_pred_best))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt = 'd', cmap = 'Blues')
plt.title('confusion matrix')
plt.show()


In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending = False)

plt.figure(figsize=(10, 8))
sns.barplot(data = feature_importance.head(15), x = 'importance', y = 'feature')
plt.title('Top 15 Feature Importance')
plt.show()